# 03 - Inference
Uses the saved model (models/model.joblib). If it is missing, the first cell downloads the data and trains it, so this notebook also runs on its own.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("working folder:", os.getcwd())
from src.data import CLEAN, prepare
if not CLEAN.exists():
    print("no processed data yet, downloading and cleaning it now")
    prepare()
from pathlib import Path
if not Path("models/model.joblib").exists():
    print("no saved model yet, training it now (about a minute)")
    from src import train
    train.main()
print("data ready:", CLEAN)

In [ ]:
import joblib
import pandas as pd
from src.data import CLEAN, NUM, CAT, split_xy
model = joblib.load("models/model.joblib")
print("model loaded:", type(model.named_steps["clf"]).__name__)

## One patient

In [ ]:
patient = {"age": 63, "sex": 1, "cp": 1, "trestbps": 145, "chol": 233, "fbs": 1,
           "restecg": 2, "thalach": 150, "exang": 0, "oldpeak": 2.3, "slope": 3,
           "ca": 0, "thal": 6}
row = pd.DataFrame([patient])[NUM + CAT].astype(float)
p = float(model.predict_proba(row)[0, 1])
print("probability of disease:", round(p, 3), "-> prediction:", int(p >= 0.5))

## A patient with ca and thal unknown
The model still works because missing values were part of training.

In [ ]:
row2 = row.copy()
row2[["ca", "thal"]] = float("nan")
p2 = float(model.predict_proba(row2)[0, 1])
print("probability of disease:", round(p2, 3))

## Five rows from the data with their true label

In [ ]:
df = pd.read_csv(CLEAN)
X, y = split_xy(df)
sample = X.sample(5, random_state=1)
out = pd.DataFrame({"source": df.loc[sample.index, "source"],
                    "true": y[sample.index],
                    "prob_disease": model.predict_proba(sample)[:, 1].round(3)})
out["predicted"] = (out.prob_disease >= 0.5).astype(int)
out

## Same prediction through the API (optional)
Start the API in a terminal first: `uvicorn src.app:app --port 8000`

In [ ]:
import json, urllib.request
req = urllib.request.Request("http://localhost:8000/predict", json.dumps(patient).encode(),
                             {"Content-Type": "application/json"})
try:
    print(json.loads(urllib.request.urlopen(req, timeout=5).read()))
except OSError as e:
    print("API is not running:", e)